# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from scipy.stats import gaussian_kde
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # or MinMaxScaler
from sklearn.model_selection import GridSearchCV
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator) 
from joblib import dump, load
from matplotlib import rc

# Data pre-processing

In [ ]:
# Create a loop that stores each data set into a dictionary
output = 'gisaxs_domain'
data_directory = '../../../pre-processing/full_descriptors/poly_maccs.xlsx'
df_likeness = pd.read_excel(data_directory)

In [ ]:
df_likeness

# Bayesian-Search

In [ ]:
def bayes_train(df, output):

    df_clean = df.dropna()

    inputs = df_clean.iloc[:, 1:-2].copy()
    target = df_clean[f"{output}"]

    X_train, X_test, y_train, y_test = train_test_split(
        inputs, target, test_size=0.2, random_state=10
    )
    
    # Scale the inputs (fit only on training data to avoid data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled = scaler.transform(X_test)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

    RANDOM_SEED = 42
    CV_FOLDS = 5

    search_spaces = {
            "n_estimators": Integer(100, 1000),
            "max_depth": Integer(3, 50),
            "min_samples_split": Integer(2, 50),
            "min_samples_leaf": Integer(1, 30),
            "min_weight_fraction_leaf": Real(0.0, 0.1, prior="uniform"),
            "min_impurity_decrease": Real(0.0, 0.05, prior="uniform"),
            "max_features": Categorical(["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8, 1.0]),
            "max_leaf_nodes": Categorical([None, 31, 63, 127, 255, 511]),
            "criterion": Categorical(["squared_error", "friedman_mse", "absolute_error", "poisson"]),
            "ccp_alpha": Real(0.0, 0.05, prior="uniform"),
            "bootstrap": Categorical([True]),          # locked
            "max_samples": Categorical([None, 0.5, 0.7, 0.8, 0.9]),  # valid here
        }
    
    rf = RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1,
        warm_start=False,   # BayesSearchCV manages refits; keep this off
    )
    
    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=60,                # more params → more iterations needed
        scoring="r2",
        cv=CV_FOLDS,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        refit=True,
    )

    # Fit optimizer
    opt.fit(X_train_scaled, y_train)

    best_model = opt.best_estimator_
    best_cv_r2 = opt.best_score_
    best_test_r2 = best_model.score(X_test_scaled, y_test)

    print("CV Train R²:", best_cv_r2)
    print("Test R²:", best_test_r2)
    print("Best Parameters:", opt.best_params_)

    return best_model, scaler

In [ ]:
%%time

best_model, scaler = bayes_train(df_likeness, output)

In [ ]:
df_likeness_key = "poly_maccs"      
dump(best_model , f'./likeness/{output}_{df_likeness_key}.joblib')

In [ ]:
validation_path = '../../../pre-processing/full_descriptors/validation/poly_maccs.xlsx'
# output = 'gisaxs_domain'
# import data frame for pre-processing
df_val = pd.read_excel(validation_path)
# Create a new DataFrame without the last row
df_val = df_val.iloc[:-1]
df_val

In [ ]:
# =========================
# Validation set evaluation
# =========================
output='gisaxs_domain'
# Drop missing values in validation set
df_val_clean = df_val.dropna()
sample_names = df_val_clean["Sample"]
# Assign inputs and target (must match training exactly)
X_val = df_val_clean.iloc[:, 1:-2]  # enforce same columns & order as training
y_val = df_val_clean[f'{output}']

# Scale using TRAINING scaler (no refitting)
X_val_scaled = scaler.transform(X_val)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)

# Predictions
y_val_pred = best_model.predict(X_val_scaled)

# Metrics
val_r2 = best_model.score(X_val_scaled, y_val)
val_mae = mean_absolute_error(y_val, y_val_pred)

print(f'Validation r2: {val_r2}, validation mae: {val_mae}')

In [ ]:
def plot_pred_vs_actual(X_test, y_test, y_test_pred, 
                        test_r2, model_output, output_path=None):

    errors = y_test_pred - y_test
    # Metrics
    me = np.mean(errors)
    mae = np.mean(np.abs(errors))
    mape = np.mean(np.abs(errors / y_test)) * 100
    r2 = 1 - np.sum(errors**2) / np.sum((y_test - np.mean(y_test))**2)
    
    # Configure figure aesthetics
    rc('text', usetex=False)
    rc('mathtext', fontset='cm')
    rc('xtick', labelsize=25)
    rc('ytick', labelsize=25)
    rc('axes', labelsize=25, linewidth=2, labelweight='500')
    rc('font', family='Arial', size=15, weight='500', style='normal')
    plt.rcParams['axes.spines.right'] = True
    plt.rcParams['axes.spines.top'] = True

    fig, ax = plt.subplots(figsize=(8, 8))

    # Plot perfect prediction line
    lims = [y_test.min()*1, y_test.max()*1]
    # Add a proportional buffer
    buffer = 0.5 * (lims[1] - lims[0])
    lims = [lims[0] - buffer, lims[1] + buffer]
    ax.plot((lims), lims, '--', color='red', zorder=0)

    # Combine datasets
    X_all = X_test
    y_all = y_test
    y_all_pred = y_test_pred
    
    colors = {'BCP': 'indigo'}
    
    # Scatter points colored by composition type
    for label, color in colors.items():
        ax.scatter(
            y_all,
            y_all_pred,
            color=color,
            edgecolors='black',
            alpha=0.75,
            s=100,
            label=label
        )

    # Aesthetic adjustments
    ax.set_xlim(0,90) #Adjusted for plotting consistency with FWHM plots
    ax.set_ylim(0,90)

    ax.set_xlabel('Actual \n domain spacing (nm)', fontsize=30)
    ax.set_ylabel('Predicted \n domain spacing (nm)', fontsize=30)
    ax.tick_params(axis='both', which='both', reset=True, top=False, right=False)
    ax.set_facecolor('white')
    ax.grid(False)

    # Error metric text
    ax.text(0.425, 0.17, 
            f'R$^2$ = {r2:.3f}\nMAE = {mae:.2f} nm\nMAPE = {mape:.2f} %', 
                 transform=ax.transAxes,
                 fontsize=20, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0))

    # Axes borders
    for ax in [ax]:
        for spine in ax.spines.values():
            spine.set_color('black')
    # Manually set the ticks and labels
    ticks = [20, 40, 60, 80]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks])
    ax.set_yticklabels([str(t) for t in ticks])

    plt.tight_layout()       
    # Save figure
    if output_path:
        plt.savefig(f"{output_path}{model_output}_parity.pdf", format='pdf', bbox_inches='tight', pad_inches=0.05,
                   transparent=True)
        # plt.savefig(f"{output_path}{model_output}_parity.png", format='png',
        #            transparent=False)

In [ ]:
plot_pred_vs_actual(X_val_scaled, y_val, y_val_pred, 
                    val_r2, "validation_domain__full", output_path=f"./")